In [ ]:
from pipeline.models import RealTimePatchCore
from pipeline.visualization import create_history_html
from pipeline.detection import yolo_edge,showDetection
from pipeline.processors import threaded_gpto1_worker
from pipeline.utilss import preprocess_for_display, log_gpu_memory
import torch
import gc
import numpy as np
import cv2
import time
import queue
import threading
import io
import traceback

from PIL import Image as PILImage
from IPython.display import display, HTML
import ipywidgets as widgets
from ultralytics import YOLO

def process_video(
    model_path,
    openai_api_key,  
    video_path=None,
    use_camera=False,
    camera_index=0,
    low_threshold=0.3,
    medium_threshold=0.5,
    high_threshold=0.7,
    object_threshold=0.5,
    detection_area_ratio=0.3,
    system_prompt="How would you describe the anomalies seen on the following paperclip?.",
    min_frames_between_detections=70,
    playback_speed=1.0,
    max_history=5,  
    enable_gpt4=False,
    disable_video_display=False,
    process_every_n_frames=1,  
):
   

    patchcore_model = RealTimePatchCore(model_path=model_path)
    yolo_model = YOLO(r"./yolo11n.pt")
    yolo_model.to("cuda")
    print("models loaded")
    class rt_state:
        def __init__(self):
            self.paused = False
            self.shutdown = False
            self.current_speed = playback_speed
            #self.frame_count = 0
            self.total_frames = 0
            # change during runtime with ui
            # self.current_low_threshold = low_threshold
            # self.current_medium_threshold = medium_threshold
            # self.current_high_threshold = high_threshold
            
#     threshold_controls = widgets.VBox([
#     widgets.HTML("<h3>Threshold Controls</h3>"),
#     widgets.FloatSlider(
#         value=low_threshold,
#         min=0.0,
#         max=1.0,
#         step=0.01,
#         description='Low:',
#         readout_format='.2f',
#         style={'description_width': '60px'}
#     ),
#     widgets.FloatSlider(
#         value=medium_threshold,
#         min=0.0,
#         max=1.0,
#         step=0.01,
#         description='Medium:',
#         readout_format='.2f',
#         style={'description_width': '60px'}
#     ),
#     widgets.FloatSlider(
#         value=high_threshold,
#         min=0.0,
#         max=1.0,
#         step=0.01,
#         description='High:',
#         readout_format='.2f',
#         style={'description_width': '60px'}
#     )
# ])

    frame_queue = queue.Queue(maxsize=100)
    result_queue = queue.Queue()
    gpt_input_q = queue.Queue(maxsize=5)  
    gpt_output_q = queue.Queue()
    video_frame_queue = queue.Queue(maxsize=100)
    detection_history = []
    detection_display_history = []
    #previous_frame = None 
    print("queues created")

    log_widget = widgets.Output(layout={
    'border': '1px solid #ccc', 
    'padding': '10px',
    'max_height': '200px', 
    'overflow_y': 'auto'
    })

    output = widgets.Output()
    video_widget = widgets.Image(format='jpeg', width=320)
    log_widget = widgets.Output(layout={
        'border': '1px solid #ccc', 
        'padding': '10px',
        'max_height': '200px', 
        'overflow_y': 'auto',
        'width': '320px'  
    })
    results_widget = widgets.Output()
    history_widget = widgets.Output()
    gpto1_widget = widgets.Output()

    left_column = widgets.VBox([
        widgets.HTML("<h3>Video Feed</h3>"),
        video_widget,
        widgets.HTML("<h3>Log Output</h3>"),
        log_widget
    ])


    right_column = widgets.VBox([
        widgets.HTML("<h3>Detection History</h3>"),
        history_widget
    ])

    layout = widgets.HBox([left_column, right_column])
  
    stats_widget = widgets.HTML(
    value="Initializing...",
    layout=widgets.Layout(
        background_color='#2e2e2e',
        border='1px solid #444',
        padding='10px',
        border_radius='10px',
        width='100%'
    )
)

    display(stats_widget)
    display(layout)
    display(results_widget)
    display(output)

    
    if enable_gpt4 and openai_api_key:
        with gpto1_widget:
            gpto1_widget.clear_output()

    else:
        with gpto1_widget:
            gpto1_widget.clear_output()

    
 
    def video_playback_thread(cap, shared_state, video_frame_queue):
        frame_count = 0
        fps = cap.get(cv2.CAP_PROP_FPS)

        
        frames_delivered = 0
        start_time = time.time()
        last_report_time = start_time
        
        while not shared_state.shutdown:
            if shared_state.paused:
                time.sleep(0.1)
                continue
                
            ret, frame = cap.read()
            if not ret:
                time.sleep(0.5)
                shared_state.shutdown = True
                break
                
            #frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
            frame = cv2.transpose(frame)
            frame = cv2.flip(frame, 1)
            frame_count += 1
            frames_delivered += 1
            

            current_time = time.time()
            if current_time - last_report_time >= 1.0:
                elapsed = current_time - last_report_time
                actual_fps = frames_delivered / elapsed
                print(f"Actual delivery rate: {actual_fps:.1f} FPS")
                frames_delivered = 0
                last_report_time = current_time
            
            
            try:
                video_frame_queue.put((frame_count, frame), block=False)
            except queue.Full:
                
                pass
    
    def process_frames_thread():
        with log_widget:
         print("Processing thread started")
        #  for i in range(5):
        #         print(i)
                
        while True:
            try:
                item = frame_queue.get(timeout=5.0)
                if item is None:
                 with log_widget:
                    print("Processing thread received shutdown signal")
                    break
                
                frame_id, timestamp, frame = item
                
                start_time = time.time()
                try:
                    score, anomaly_mask = patchcore_model.process_frame(frame)
                    resize_dim = 256  
                    crop_dim = 244   
                    display_frame = preprocess_for_display(frame, resize_dim, crop_dim)
                    
                    heatmap = patchcore_model.generate_heatmap(
                        anomaly_mask, 
                        (crop_dim, crop_dim, 3),  
                        anomaly_score=score,
                        preserve_original_size=True,
                        target_size=(crop_dim, crop_dim)
                    )
                    
                    is_anomaly = score > low_threshold
                    severity = "normal"
                    status = "NORMAL"
                    
                    if score > high_threshold:
                        severity = "high"
                        status = "HIGH ANOMALY"
                    elif score > medium_threshold:
                        severity = "medium"
                        status = "MEDIUM ANOMALY"
                        heatmap = cv2.convertScaleAbs(heatmap, alpha=0.9, beta=0)
                    elif score > low_threshold:
                        severity = "low"
                        status = "LOW ANOMALY"
                        heatmap = cv2.convertScaleAbs(heatmap, alpha=0.7, beta=0)
                    else:
                        # If not anomaly, remove heatmap
                        heatmap = np.zeros_like(heatmap)
                    
                    display_frame_resized = cv2.resize(display_frame, (resize_dim, resize_dim))
                    heatmap_resized = cv2.resize(heatmap, (resize_dim, resize_dim))
                    
  
                    alpha = 0.6
                    overlay = cv2.addWeighted(display_frame_resized, 1-alpha, heatmap_resized, alpha, 0)
                    process_time = time.time() - start_time
                    result = {
                        'frame_id': frame_id,
                        'timestamp': timestamp,
                        'score': float(score),
                        'is_anomaly': bool(is_anomaly),
                        'severity': severity,  
                        'process_time': process_time,
                        'heatmap': heatmap_resized,
                        'heatmap_display': heatmap_resized.copy(),
                        'original_frame': display_frame_resized,
                        'raw_frame': frame.copy(),
                        'overlay': overlay,
                        'status': status
                    }
                    if is_anomaly:
                        try:
                            if enable_gpt4 and openai_api_key:
                                gpto1_item = result.copy()
                                gpto1_item['detection_id'] = f"frame_{frame_id}"
                                gpt_input_q.put(gpto1_item, block=False)
                        except queue.Full:
                            print(f"gpt input q full")
 
                    result_queue.put(result)
                except Exception as e:
                    traceback.print_exc()
                    result_queue.put({
                        'frame_id': frame_id,
                        'timestamp': timestamp,
                        'error': str(e),
                        'status': 'ERROR'
                    })
                frame_queue.task_done()
                
            except queue.Empty:
                continue
            except Exception as e:
                traceback.print_exc()
    

    def gpto1_result_thread():
        print("GPT result thread running")
        while True:
            try:
                result = gpt_output_q.get(timeout=1.0)
                if result is None:
                    break
                gpto1_result_text = result.get('gpto1_result', '')
                for i, item in enumerate(detection_display_history):
                    if item.get('frame_id') == frame_id:
                        detection_display_history[i]['gpto1_result'] = gpto1_result_text
                        with history_widget:
                            history_widget.clear_output(wait=True)
                            html_content = create_history_html(detection_display_history)
                            display(HTML(html_content))
                        break
                
            except queue.Empty:
                continue
            except Exception as e:
                print(f"{str(e)}", flush=True)
                traceback.print_exc()
            
    process_thread = threading.Thread(target=process_frames_thread)
    process_thread.daemon = True
    process_thread.start()
    print("Processing thread started")
    if enable_gpt4 and openai_api_key:
        gpto1_thread = threading.Thread(
            target=threaded_gpto1_worker, 
            args=(gpt_input_q, gpt_output_q, openai_api_key, system_prompt)
        )
        gpto1_thread.daemon = True
        gpto1_thread.start()

        gpto1_result_thread = threading.Thread(target=gpto1_result_thread)
        gpto1_result_thread.daemon = True
        gpto1_result_thread.start()
    else:
        with gpto1_widget:
            gpto1_widget.clear_output()
            print("GPT-o1 Analysis disabled")
    
    def result_thread():
        #latest_result = None
        #anomaly_count = 0
        
        
        
        all_results = []
        
        
        while True:
            try:
                result = result_queue.get(timeout=1.0)
                
                if result is None:
                    break
                #latest_result = result
                
                
                if 'heatmap' in result:
                    heatmap = result['heatmap']
                    if len(heatmap.shape) == 2: 
                        heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_GRAY2RGB)
                    else:  
                        heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
                    square_size = 244
                    heatmap_resized = cv2.resize(heatmap_rgb, (square_size, square_size), interpolation=cv2.INTER_NEAREST)
                    pil_img = PILImage.fromarray(heatmap_resized)
                    buf = io.BytesIO()
                    
                    
                    pil_img.save(buf, format='PNG')
                    if 'original_frame' in result:
                        detection_display_history.append(result)
                        if len(detection_display_history) > max_history:
                            detection_display_history.pop(0)

                        with history_widget:
                            history_widget.clear_output()
                            html_content = create_history_html(detection_display_history)
                            display(HTML(html_content))

                all_results.append(result)

            except queue.Empty:
                time.sleep(0.01)
                continue
                
    result_thread = threading.Thread(target=result_thread)
    result_thread.daemon = True
    result_thread.start()
    print("result thread started")
    try:
        if use_camera:
            cap = cv2.VideoCapture(camera_index)
            # source_type = "camera"
            if not cap.isOpened():
                with output:
                    print("could not open camera")
                return
            total_frames = 0  
            
            
            
        else:
            if not video_path:
                with output:
                    print("Error: No video file path provided.")
                return
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                with output:
                    print(f"Error: Could not open video file: {video_path}")
                return
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
        # with output:
        #     output.clear_output(wait=True)
        #     if use_camera:
        #         print(f"Input Source: Live Camera (index {camera_index})")
        #         print(f"FPS : {fps:.1f}")
        #     else:
        #         print(f"Input Source: Video File ({video_path})")
        #         print(f"FPS: {fps:.1f}")

        #fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        shared_state = rt_state()
        shared_state.total_frames = total_frames
        shared_state.current_speed = playback_speed 

        warmup_frames = 10  
        if not use_camera:
            original_pos = cap.get(cv2.CAP_PROP_POS_FRAMES)
        for _ in range(warmup_frames):
            ret, warmup_frame = cap.read()
            if not ret:
                break
            process_scale = .25
            h, w = warmup_frame.shape[:2]
            process_width = int(w * process_scale)
            process_height = int(h * process_scale)
            processing_frame = cv2.resize(warmup_frame, (process_width, process_height))
            
           
            with torch.no_grad():  
                _ = yolo_model.predict(source=processing_frame, verbose=False)
                _, _ = patchcore_model.process_frame(processing_frame)
    
            torch.cuda.empty_cache()
        if not use_camera:
            cap.set(cv2.CAP_PROP_POS_FRAMES, original_pos)

        if 'video_frame_queue' not in locals():
            video_frame_queue = queue.Queue(maxsize=100)  
        
        video_thread = threading.Thread(target=video_playback_thread, args=(cap, shared_state, video_frame_queue))
        video_thread.daemon = True
        video_thread.start()
        
   
        detected_object_count = 0
        start_time = time.time()
        current_object_threshold = object_threshold
        
        frames_since_last_detection = min_frames_between_detections
        
        # Main video processing loop 
        
        while not shared_state.shutdown or not video_frame_queue.empty():
            try:

                frame_id, frame = video_frame_queue.get(timeout=1.0)
                frame_count = frame_id 
                
                frames_since_last_detection += 1
                can_detect = frames_since_last_detection >= min_frames_between_detections
                should_process = (frame_count % process_every_n_frames == 0)
                progress = frame_count / total_frames if total_frames > 0 else 0
           
                process_scale = .25
                actual_height, actual_width = frame.shape[:2]
                process_width = int(actual_width * process_scale)
                process_height = int(actual_height * process_scale)
                processing_frame = cv2.resize(frame, (process_width, process_height))
                #print("test 1")

                if not disable_video_display:
                    frame_height, frame_width = processing_frame.shape[:2]  
                    display_scale = 1
                    display_width = int(frame_width * display_scale)
                    display_height = int(frame_height * display_scale)
                    
                    video_display = cv2.resize(processing_frame, (display_width, display_height))
                    
                if can_detect and should_process:
                    try:
                        #print("test")
                        is_centered, confidence, bbox, detection_info = yolo_edge(
                            processing_frame,
                            model=yolo_model,  
                            center_tolerance=detection_area_ratio, 
                            edge_detection_threshold=80,  # For edge detection 
                            min_contour_area=5,    # Allow small contours
                            max_contour_area=15000, # Maximum paperclip size
                            confidence_threshold=current_object_threshold,  # For YOLO detection
                            history_buffer=detection_history,
                            debug_mode=False,
                            use_gpu=True,
                        )
   
                        if not disable_video_display and is_centered and confidence > 0:
                    
                            scaled_bbox = (
                                int(bbox[0] * display_scale),
                                int(bbox[1] * display_scale),
                                int(bbox[2] * display_scale),
                                int(bbox[3] * display_scale)
                            )
                            
                            display_info = detection_info.copy()
                            if 'box_width' in display_info:
                                display_info['box_width'] = int(display_info['box_width'] * display_scale)
                            if 'box_height' in display_info:
                                display_info['box_height'] = int(display_info['box_height'] * display_scale)
                            
                            video_display = showDetection(
                                video_display,  
                                is_centered,    
                                scaled_bbox,    
                                display_info,   
                                thickness=2,    
                                text_scale=0.5  
                            )
                        
                        if is_centered and confidence >= 0.1:
                            with output:
                                method = detection_info.get('method', 'unknown')
                                print(f"PAPERCLIP CENTERED at frame {frame_count} using {method}")
                                print(f"- Confidence: {confidence:.2f}")
                            timestamp = time.time()
                            try:
                                frame_queue.put((frame_count, timestamp, processing_frame.copy()), block=False)
                                detected_object_count += 1
                                frames_since_last_detection = 0
                            except queue.Full:
                                with output:
                                    print("process queue full")
                    except Exception as e:
                        with output:
                            traceback.print_exc()
                if frame_count % 1 == 0 and not disable_video_display:
                    rgb_video = cv2.cvtColor(video_display, cv2.COLOR_BGR2RGB)
                    pil_img = PILImage.fromarray(rgb_video)
                    buf = io.BytesIO()
                    pil_img.save(buf, format='JPEG')
                    video_widget.value = buf.getvalue()
        
                elapsed_time = time.time() - start_time
                process_fps = frame_count / elapsed_time if elapsed_time > 0 else 0
                queue_size = frame_queue.qsize()
                
                stats_html = f"""
                <div style="font-family: monospace; background-color: #2e2e2e; color: #ffffff; padding: 10px; border-radius: 10px;">
                    <div style="font-weight: bold; font-size: 18px;">Status: Processing</div>
                    <div>Progress: {frame_count}/{total_frames} ({progress*100:.1f}%)</div>
                    <div>Playback FPS: {process_fps:.1f}</div>
                    <div>Objects Detected: {detected_object_count}</div>
                    <div>Queue Size: {queue_size}</div>
                    <div>Thresholds: 
                        <span style="color: orange;">Low={low_threshold}</span>, 
                        <span style="color: red;">Medium={medium_threshold}</span>, 
                        <span style="color: darkred;">High={high_threshold}</span>,
                        Object={current_object_threshold}
                    </div>
                    <div>Playback Speed: {shared_state.current_speed}x</div>
                    <div>Showing last {len(detection_display_history)}/{max_history} detections in history</div>
                </div>
                """
                stats_widget.value = stats_html
            except queue.Empty:
                continue    
    except KeyboardInterrupt:
        with output:
            print("Interrupted by user")
    except Exception as e:
        with output:
            print(f"Error: {str(e)}")
            traceback.print_exc()
    finally:
            print("waiting for gpt analysis")
            wait_time = 30 
            if enable_gpt4 and openai_api_key:
                pending = gpt_input_q.qsize() + gpt_output_q.qsize()
                if pending > 0:
                    stats_html = f"""
                    <div style="font-family: monospace; background-color: #2e2e2e; color: #ffffff; padding: 10px; border-radius: 10px;">
                        <div style="font-weight: bold; font-size: 18px;">Status: Waiting for pending analyses</div>
                        <div>Waiting up to {wait_time} seconds for {pending} GPT-4V tasks to complete...</div>
                        <div>Objects Detected: {detected_object_count}</div>
                    </div>
                    """
                    stats_widget.value = stats_html
                    start_wait = time.time()
                    while time.time() - start_wait < wait_time:
                        remaining = gpt_input_q.qsize() + gpt_output_q.qsize()
                        if remaining == 0:
                            break
                        print(f"waiting")
                        time.sleep(1)
                    else:
                        print("timeout")

            
            
            shared_state.shutdown = True 
            frame_queue.put(None)
            result_queue.put(None)
            if enable_gpt4 and openai_api_key:
                gpt_input_q.put(None) 
                gpt_output_q.put(None) 
            time.sleep(2)
            def clear_queue(q, name):
                count = 0
                try:
                    while not q.empty():
                        q.get_nowait()
                        count += 1
                except:
                    pass
                if count > 0:
                    print(f"Cleared {count} items from {name}")
            clear_queue(frame_queue, "frame queue")
            clear_queue(result_queue, "result queue")
            clear_queue(gpt_input_q, "GPT input queue")
            clear_queue(gpt_output_q, "GPT output queue")
            clear_queue(video_frame_queue, "video frame queue")  
            torch.cuda.empty_cache()
            gc.collect()

            if 'cap' in locals() and cap.isOpened():
                cap.release()
            stats_html = f"""
            <div style="font-family: monospace; background-color: #2e2e2e; color: #ffffff; padding: 10px; border-radius: 10px;">
                <div style="font-weight: bold; font-size: 18px;">Status: Complete</div>
                <div>Total Objects Detected: {detected_object_count}</div>
                <div>Completed successfully.</div>
            </div>
            """
            stats_widget.value = stats_html 
            print("All resources released, shutdown complete")